# Reproduce: (216) Kleopatra: asteroid motion against JPL Horizons

Two modes, set in the next cell:

* `MODE = "offline"` (default): never touches the network. Recomputes from the FITS fixtures in
  `backend/tests/fixtures` where the case has them, replays the published analysis from cutouts already in
  `data/cache`, and checks every cached file against the sha256 in the bundle manifest.
* `MODE = "online"`: re-downloads from IRSA (and reruns the analysis live). Slow; a checksum that changed is
  reported as DRIFT, i.e. the archive reprocessed the product after we published.

Numbers printed as "published" come from `submission/evidence`; nothing here edits them.

In [1]:
CASE = "kleopatra"
MODE = "offline"          # "offline" or "online"
MAX_ONLINE_ITEMS = 5      # archive items to re-download in online mode (None = all)
FULL_REPLAY = True        # rerun the whole analysis (offline: from data/cache only)

In [2]:
import json, sys, zipfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend" / "hipparchus").exists())
sys.path.insert(0, str(ROOT / "backend"))
from hipparchus import bundle

spec = bundle.CASES[CASE]
ev = json.loads((bundle.EVIDENCE / spec["evidence"]).read_text())
zpath = bundle.BUNDLE_DIR / f"{CASE}.zip"
man = json.loads(zipfile.ZipFile(zpath).read(f"{CASE}/manifest.json"))
print(spec["title"])
print("bundle", zpath.relative_to(ROOT), "| evidence sha256", man["evidence_files"][0]["sha256"][:16], "...")
print("datasets", [d["doi"] for d in man["datasets"]], "| cutouts", len(man["cutouts"]),
      "| archive queries", len(man["archive_queries"]))

(216) Kleopatra: motion of a known asteroid against JPL Horizons
bundle submission/bundles/kleopatra.zip | evidence sha256 e9ec2b867f03e988 ...
datasets ['10.26131/IRSA678'] | cutouts 38 | archive queries 2


In [3]:
fm = ev["fitted_motion"]
print("published rate (arcsec/h):", round(fm["measured_rate_arcsec_h"]["ra_cosdec"], 4), round(fm["measured_rate_arcsec_h"]["dec"], 4))
print("Horizons, same model     :", round(fm["horizons_same_model_rate_arcsec_h"]["ra_cosdec"], 4), round(fm["horizons_same_model_rate_arcsec_h"]["dec"], 4))
print("published residuals      :", {k: round(v, 3) for k, v in ev["residuals_vs_horizons_spherex_arcsec"]["all_detections_near_prediction"].items()})

published rate (arcsec/h): 2.0767 4.3102
Horizons, same model     : 2.0771 4.3075
published residuals      : {'n': 31, 'mean_ra_cosdec': -0.079, 'mean_dec': 0.023, 'rms_ra_cosdec': 0.147, 'rms_dec': 0.185, 'median_total': 0.207, 'max_total': 0.496}


In [4]:
# Checksums: cache in offline mode, the archive itself in online mode.
r = bundle.verify(zpath, network=(MODE == "online"), limit=MAX_ONLINE_ITEMS if MODE == "online" else None)
print(json.dumps(r["summary"]))
for i in r["items"]:
    if i["status"] in ("DRIFT", "unreachable"):
        print(i["status"], i["url"][:110], i.get("note", i.get("error", "")))
print("bundle OK" if r["ok"] else "bundle NOT OK")

{"bundle_files": {"match": 7}, "data": {"match": 40, "no_checksum": 2}, "source_files": {"match": 5, "changed_since_bundle": 1}}
bundle OK


In [5]:
# Offline: recompute the residual summary from the per-exposure numbers in the evidence.
# (The full blind search, scripts/reproduce_motion.py, takes ~17 min with Horizons + IRSA.)
import math
rows = []
for e in ev["exposures"]:
    m, h = e.get("measured"), e.get("horizons_spherex")
    if m and h and m.get("ra") is not None:
        rows.append(((m["ra"] - h["ra"]) * math.cos(math.radians(h["dec"])) * 3600, (m["dec"] - h["dec"]) * 3600))
rms = lambda v: math.sqrt(sum(x * x for x in v) / len(v))
pub = ev["residuals_vs_horizons_spherex_arcsec"]["all_detections_near_prediction"]
print("n", len(rows), "published", pub["n"])
print("rms RA cosDec", round(rms([r[0] for r in rows]), 4), "published", round(pub["rms_ra_cosdec"], 4))
print("rms Dec      ", round(rms([r[1] for r in rows]), 4), "published", round(pub["rms_dec"], 4))
if MODE == "online" and FULL_REPLAY:
    import subprocess
    subprocess.run([sys.executable, str(ROOT / "backend" / "scripts" / "reproduce_motion.py")], check=True)

n 31 published 31
rms RA cosDec 0.1471 published 0.1471
rms Dec       0.1846 published 0.1846
